# Causal attention and dropout

This notebook continues from `attention.ipynb`. It turns single-head
self-attention into the version used in a GPT model, with three changes:

1. A **causal mask**: each token can attend only to itself and to the tokens
   before it, never to the tokens after it.
2. **Dropout** on the attention weights, to reduce overfitting during training.
3. A **batch dimension**: the input is a batch of `b` sequences, with shape
   `(b, num_tokens, d_in)`, instead of a single sequence.

The example follows the causal attention section in Chapter 3 of
*Build a Large Language Model (From Scratch)* by Sebastian Raschka.

> **Origin and license.** This notebook is derived from [`ch03/01_main-chapter-code/ch03.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb) in the repository [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Copyright 2023-2026 Sebastian Raschka, licensed under the [Apache License 2.0](../LICENSES/Apache-2.0-LLMs-from-scratch.txt).
>
> **Modified by L. Saetta (2026):** explanations rewritten and extended, and code adapted to this repository. See [NOTICE](../NOTICE).

In [1]:
import torch
from torch import nn

## 1. Why a causal mask?

A GPT model is trained to **predict the next token**. For the sequence
"Your journey starts with one step", the training targets are:

| Input so far | Token to predict |
| --- | --- |
| Your | journey |
| Your journey | starts |
| Your journey starts | with |
| ... | ... |

In unmasked self-attention, every token attends to **all** the tokens of the
sequence, including the ones after it. During training, the model could then
"read the answer": the representation of "journey" would contain information
about "starts", which is exactly the token it must predict.

The causal mask removes this shortcut. Token `i` may attend only to tokens
`0, 1, ..., i`. In the attention weight matrix, rows are queries and columns
are keys, so every weight **above the main diagonal** must be zero.

We use the same six illustrative input vectors as in the previous notebooks.

In [2]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],  # step     (x^6)
    ]
)

num_tokens, d_in = inputs.shape
d_out = d_in

## 2. Build the mask step by step

First, we compute the attention scores as in `attention.ipynb`, with
query and key projections. The scores are a `(6, 6)` matrix: one score for
each query–key pair, before any masking.

In [3]:
torch.manual_seed(123)
W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)

with torch.no_grad():
    queries = W_query(inputs)
    keys = W_key(inputs)
    attn_scores = queries @ keys.T

print(attn_scores)

tensor([[-0.4028, -0.2063, -0.2069, -0.0635, -0.1611, -0.0672],
        [-0.2623,  0.1610,  0.1602,  0.1450,  0.1019,  0.1546],
        [-0.2630,  0.1553,  0.1546,  0.1416,  0.0979,  0.1510],
        [-0.0989,  0.1501,  0.1497,  0.1111,  0.1010,  0.1183],
        [-0.2004,  0.0102,  0.0098,  0.0397, -0.0013,  0.0425],
        [-0.1048,  0.2070,  0.2065,  0.1480,  0.1407,  0.1575]])


`torch.triu(..., diagonal=1)` keeps the elements **above** the main diagonal
of a matrix and sets the others to zero. Applied to a matrix of ones, it gives
a mask with `1` exactly at the positions that a token must not see: row `i`,
columns `i + 1` and later.

In [4]:
mask = torch.triu(torch.ones(num_tokens, num_tokens), diagonal=1)
print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


We now replace the masked scores with $-\infty$ before softmax. Since
$\exp(-\infty) = 0$, softmax gives those positions a weight of exactly zero,
and the remaining weights of each row still sum to one.

`masked_fill` needs a boolean mask: `True` where the value must be replaced.

In [5]:
masked_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked_scores)

tensor([[-0.4028,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2623,  0.1610,    -inf,    -inf,    -inf,    -inf],
        [-0.2630,  0.1553,  0.1546,    -inf,    -inf,    -inf],
        [-0.0989,  0.1501,  0.1497,  0.1111,    -inf,    -inf],
        [-0.2004,  0.0102,  0.0098,  0.0397, -0.0013,    -inf],
        [-0.1048,  0.2070,  0.2065,  0.1480,  0.1407,  0.1575]])


In [6]:
attn_weights = torch.softmax(masked_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)
print("Row sums:", attn_weights.sum(dim=-1))

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4392, 0.5608, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2820, 0.3591, 0.3589, 0.0000, 0.0000, 0.0000],
        [0.2253, 0.2602, 0.2601, 0.2544, 0.0000, 0.0000],
        [0.1809, 0.2043, 0.2042, 0.2078, 0.2029, 0.0000],
        [0.1456, 0.1743, 0.1743, 0.1685, 0.1678, 0.1694]])
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


The first token, "Your", can attend only to itself, so its weight is `1.0`.
The last token, "step", can attend to all six tokens.

**Why $-\infty$ before softmax, and not zero after it?** Setting the weights
to zero *after* softmax would leave rows that no longer sum to one, and we
would need to normalize them again. The two approaches give the same result,
as the next cell shows, but masking the scores with $-\infty$ does it in a
single step.

Every row keeps at least one unmasked position, the diagonal. If a whole row
were masked, softmax would divide zero by zero and return `NaN`.

In [7]:
weights_unmasked = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
lower_triangle = torch.tril(torch.ones(num_tokens, num_tokens))

weights_zeroed = weights_unmasked * lower_triangle
weights_renormalized = weights_zeroed / weights_zeroed.sum(dim=-1, keepdim=True)

print("Same result:", torch.allclose(weights_renormalized, attn_weights))

Same result: True


## 3. Dropout on the attention weights

**Dropout** is a regularization technique: during training, it sets a random
fraction `p` of the values to zero, so that the model cannot rely too much on
any single connection. It is used only during **training**; when the model is
used for inference, dropout does nothing.

In attention, dropout is applied to the attention weights: some query–key
connections are randomly removed at each training step.

To keep the expected value of the output unchanged, PyTorch multiplies the
values that are kept by `1 / (1 - p)`. With `p = 0.5`, the kept values are
doubled.

A module is in training mode after its creation. `.eval()` switches it to
evaluation mode, and `.train()` switches it back.

In [8]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
example = torch.ones(6, 6)

print("Training mode:")
print(dropout(example))

dropout.eval()
print("Evaluation mode:")
print(dropout(example))

Training mode:
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])
Evaluation mode:
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])


Applied to the causal attention weights, dropout removes some connections and
doubles the others. The rows no longer sum to one, and this is expected: it
happens only during training.

GPT-2 uses a small dropout rate, `0.1`; we use `0.5` here only to make the
effect easy to see.

In [9]:
torch.manual_seed(123)
dropout.train()
dropped_weights = dropout(attn_weights)

print(dropped_weights)
print("Row sums:", dropped_weights.sum(dim=-1))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5640, 0.7181, 0.7178, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5203, 0.5202, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4085, 0.0000, 0.4155, 0.0000, 0.0000],
        [0.0000, 0.3487, 0.3486, 0.3370, 0.3356, 0.0000]])
Row sums: tensor([2.0000, 0.0000, 2.0000, 1.0406, 0.8240, 1.3699])


In this run, the whole second row is dropped: for this training step, the
token "journey" receives no context at all. With the small dropout rates
used in practice, this is rare, and in the next step a different random set
of weights is dropped.

## 4. A batch of sequences

In training, the model processes several sequences at once, in a **batch**.
The input gets a new first dimension: `(b, num_tokens, d_in)`. Here we build
a batch of two sequences, by stacking the same input twice.

In [10]:
batch = torch.stack((inputs, inputs), dim=0)

# 2 sequences, with 6 tokens each, and 3 features per token.
print(batch.shape)

torch.Size([2, 6, 3])


## 5. The `CausalAttention` class

The class combines the steps above. Compared with `SelfAttention_v2` in
`attention.ipynb`, it has four changes (marked with `New` and `Changed` in the
code):

* **Batch support.** The keys have shape `(b, num_tokens, d_out)`, so we
  transpose only the last two dimensions, `keys.transpose(1, 2)`, instead of
  `keys.T`.
* **Causal mask, as a buffer.** The mask is built once, for the maximum
  sequence length `context_length`, and registered with `register_buffer`.
  A buffer is part of the module, like a parameter, so it moves with the
  model when we call `.to(device)`, but it is **not trainable**. In `forward`,
  `[:num_tokens, :num_tokens]` takes the part of the mask that matches the
  actual sequence length.
* **In-place masking.** `masked_fill_`, with a trailing underscore, modifies
  the score tensor in place instead of creating a new one.
* **Dropout** on the attention weights.

In [11]:
class CausalAttention(nn.Module):
    """Single-head causal self-attention with dropout, for batches of sequences.

    Args:
        d_in: Number of features in each input token embedding.
        d_out: Number of features in each query, key, value, and output vector.
        context_length: Maximum number of tokens in a sequence.
        dropout: Probability of dropping an attention weight during training.
        qkv_bias: Whether each projection includes a trainable bias.
    """

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)  # New
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )  # New

    def forward(self, x):
        """Compute one context vector per token, for each sequence of the batch.

        Args:
            x: Input embeddings with shape (b, num_tokens, d_in).

        Returns:
            Context vectors with shape (b, num_tokens, d_out).
        """
        b, num_tokens, d_in = x.shape  # New batch dimension b
        # For inputs where `num_tokens` exceeds `context_length`, this will result
        # in errors in the mask creation further below. In practice, this is not a
        # problem since the LLM (chapters 4-7 of the book) ensures that inputs do
        # not exceed `context_length` before reaching this forward method.
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)  # Changed transpose
        # New, _ ops are in-place. `:num_tokens` accounts for cases where the
        # number of tokens in the batch is smaller than the supported context size.
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)  # New

        context_vec = attn_weights @ values
        return context_vec

We create the module with `dropout = 0.0`, so that the output is
deterministic. The output has one context vector per token, for each of the
two sequences: shape `(2, 6, 3)`. The two sequences are identical, so they
give identical outputs.

In [12]:
torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, dropout=0.0)

context_vecs = ca(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.3326,  0.5659, -0.3132],
         [ 0.3456,  0.5650, -0.2237],
         [ 0.3440,  0.5604, -0.2000],
         [ 0.3103,  0.4941, -0.1606],
         [ 0.2430,  0.4287, -0.1643],
         [ 0.2648,  0.4316, -0.1375]],

        [[ 0.3326,  0.5659, -0.3132],
         [ 0.3456,  0.5650, -0.2237],
         [ 0.3440,  0.5604, -0.2000],
         [ 0.3103,  0.4941, -0.1606],
         [ 0.2430,  0.4287, -0.1643],
         [ 0.2648,  0.4316, -0.1375]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 3])


## 6. Check: the future does not influence the past

If the mask works, changing a token must not change the outputs of the tokens
**before** it. We change only the last token ("step") of the second sequence,
and compare the outputs.

In [13]:
changed_batch = batch.clone()
changed_batch[1, -1] = torch.tensor([9.0, -9.0, 9.0])

with torch.no_grad():
    original_output = ca(batch)
    changed_output = ca(changed_batch)

print(
    "Earlier tokens unchanged:",
    torch.allclose(original_output[1, :-1], changed_output[1, :-1]),
)
print(
    "Last token changed:",
    not torch.allclose(original_output[1, -1], changed_output[1, -1]),
)
print(
    "First sequence unchanged:", torch.allclose(original_output[0], changed_output[0])
)

Earlier tokens unchanged: True
Last token changed: True
First sequence unchanged: True


The same check also shows that the sequences of a batch are independent:
changing the second sequence does not affect the first one.

## 7. Dropout in training and evaluation mode

With dropout, the output of the module depends on its mode. In training
mode, each call drops a different random set of attention weights, so two
calls on the same input give different results. In evaluation mode, dropout
is disabled and the output is deterministic.

In [14]:
torch.manual_seed(123)
ca_dropout = CausalAttention(d_in, d_out, context_length, dropout=0.5)

with torch.no_grad():
    ca_dropout.train()
    print(
        "Training mode, two calls equal:",
        torch.equal(ca_dropout(batch), ca_dropout(batch)),
    )

    ca_dropout.eval()
    print(
        "Evaluation mode, two calls equal:",
        torch.equal(ca_dropout(batch), ca_dropout(batch)),
    )

Training mode, two calls equal: False
Evaluation mode, two calls equal: True


## Shapes and limits of this example

| Tensor | Shape |
| --- | --- |
| Input batch | `(b, num_tokens, d_in)` |
| Queries, keys, values | `(b, num_tokens, d_out)` |
| Attention scores and weights | `(b, num_tokens, num_tokens)` |
| Causal mask (buffer) | `(context_length, context_length)` |
| Output context vectors | `(b, num_tokens, d_out)` |

The mask has no batch dimension: PyTorch **broadcasts** the same
`(num_tokens, num_tokens)` mask over all the sequences of the batch.

Inputs longer than `context_length` are not supported: the mask would be
smaller than the score matrix, and `masked_fill_` would fail.

This is still a **single-head** attention. The next step is multi-head
attention: several causal attention heads in parallel, whose outputs are
combined by a final linear projection.